<a href="https://colab.research.google.com/github/james-weichert/ai-foundations/blob/main/ai_foundations_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# An Introduction to the Foundations and Impacts of Artificial Intelligence

### UW Allen School Admitted Students Day 2026

[**James Weichert**](https://jpw.info), Assistant Teaching Professor

**Using This Notebook**

This "notebook" combines Python code with text to guide you through the process of building a _k-nearest neighbors_ classifier, a type of machine learning (or AI) algorithm.

To run the code in this notebook, **click on each code cell and press `Shift` + `Return` on your keyboard** (or click the play button to the left of the cell).

In [10]:
# Run this cell (by pressing Shift + Return)
# to import the necessary Python libraries for this demo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go

## Seattle Restaurant Recommender &trade;

James is a **big** foodie and he loves trying local restaurants. Help James build an _algorithm_—a computer program that performs a specific task—to recommend a Seattle restaurant based on user preferences.

The `seattle_restaurants` table below has 169 restaurants from across the city, along with information about their average price (in dollars), average rating (out of 5), and number of reviews. This data is sourced from [this Kaggle dataset](https://www.kaggle.com/datasets/oklena/seattle-area-restaurants) based on Yelp reviews.

In [73]:
seattle_restaurants = pd.read_csv("https://github.com/james-weichert/ai-foundations/blob/main/seattle_restaurants.csv?raw=true")
seattle_restaurants[['Name', 'Avg. Price', 'Avg. Rating', 'Num. Reviews']]

,Name,Avg. Price,Avg. Rating,Num. Reviews
0,Tavolàta,36.96,4.21,343.0
1,Local 360,30.50,3.96,3163.0
2,Café Campagne,22.75,4.20,1153.0
3,Katsu-ya Seattle,34.66,4.36,89.0
4,Tamari Bar,28.91,4.24,392.0
...,...,...,...,...
164,Azuki Handmade Japanese Udon Noodles,29.86,4.46,101.0
165,Eden Hill Restaurant,88.17,4.82,238.0
166,Gold Bar,28.67,4.47,87.0
167,Star Fusion & Bar,28.15,5.00,124.0


Feel free to use the `distance` function in your implementation. The function takes in the 'x', 'y' and 'z' coordinates for two points and returns the [Euclidean distance](https://en.wikipedia.org/wiki/Euclidean_distance) between them.

In [110]:
def distance(x1, y1, x2, y2):
    """Return the Euclidean distance between two points (x1, y1) and (x2, y2)"""

    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

### **Try It Yourself:** Build a Restaurant Recommender &trade;

Implement the `best_match` function function, which should make a recommendation based on the closest
Seattle restaurant to the user's `ideal_price`, `ideal_rating`, and `ideal_num_reviews` preferences. The function should return the name of the restaurant in `restaurants` that best matches the user's preferences.

In [127]:
def best_match(restaurants, ideal_price, ideal_rating):
    """
    Given a table `restaurants`, an `ideal_price`, and an `ideal_rating`,
    return the name of the restaurant that is the best match
    """

    n = len(restaurants)

    smallest_dist = 1_000_000 # a very large number
    best_restaurant = "" # to keep track of the restaurant we're recommending so far...

    # WRITE YOUR CODE BELOW

    # Hint: Given an index i, you can get a row of the restaurants table
    # with the expression `restaurants.iloc[i]`

    return # return the name of the recommended restaurant

##### James' Implementation (Open to Reveal)

In [150]:
def best_match_implementation(restaurants, ideal_price, ideal_rating, standardize = False):
    """
    Given a table `restaurants`, an `ideal_price`, and an `ideal_rating`,
    return the name of the restaurant that is the best match
    """

    if standardize:
      price_avg = np.mean(restaurants['Avg. Price'])
      price_sd = np.std(restaurants['Avg. Price'])
      rating_avg = np.mean(restaurants['Avg. Rating'])
      rating_sd = np.std(restaurants['Avg. Rating'])

      # Standardize ideal price and rating
      ideal_price = (ideal_price - price_avg) / price_sd
      ideal_rating = (ideal_rating - rating_avg) / rating_sd
    else:
      price_avg, rating_avg = 0
      price_sd, rating_sd = 0

    n = len(restaurants)

    smallest_dist = 1_000 # a very large number
    best_restaurant = "" # to keep track of the restaurant we're recommending so far...

    # Loop over each index in the table
    for i in range(n):

        restaurant = restaurants.iloc[i]

        price = (restaurant['Avg. Price'] - price_avg) / price_sd
        rating = (restaurant['Avg. Rating'] - rating_avg) / rating_sd

        # Find the Euclidean distance using the `distance` function
        new_dist = distance(price, rating, ideal_price, ideal_rating)

        # If the new distance is smaller than what we've seen so far,
        # update the smallest distance and keep track of the new restaurant name
        if new_dist < smallest_dist:

            smallest_dist = new_dist
            best_restaurant = restaurant['Name']

    return best_restaurant # return the name of the restaurant with the smallest distance

### Testing Our Algorithm

In the cell below, assign your ideal price and rating values to the `ideal_price` and `ideal_rating` values to see your ***Restaurant Recommender&trade;*** in action!

If your `best_match` function returns the _name_ of a restaurant in the `seattle_restaurants` table, you should see your recommended restaurant outputted below the cell.

In [151]:
# What are your ideal values for price, rating, and number of reviews?

ideal_price = 0 # replace with your value
ideal_rating = 0 # replace with your value

best_match(seattle_restaurants, ideal_price, ideal_rating)

### Visualizing Recommendations

Use the two cells below to visualize how the ***Restaurant Recommender&trade;*** changes based on your desired price and rating, which you can change using the two sliders. After selecting your price and rating, run the second cell to generate a scatter plot.

In [184]:
# This cell takes care of setting up interactive sliders to change the `price` and `rating`
# and visualizing the restaurants as points on a scatter plot

# JUST RUN THIS CELL

price = 20
rating = 4.2

def plot_restaurants(restaurants, price, rating, square_aspect = False):

  recommended_restaurant = best_match(restaurants, price, rating)
  colors = restaurants['Name'] == recommended_restaurant

  fig = px.scatter(restaurants, x = 'Avg. Price', y = 'Avg. Rating', hover_name = 'Name', color = colors)
  fig.add_trace(go.Scatter(x = [price], y = [rating], name='My Preference'))

  fig.update_traces(marker = {'size': 10})
  fig.update_layout(showlegend = False, title = 'Restaurant Recommender')

  if square_aspect:
    fig.update_layout(yaxis_scaleanchor="x")

  fig.show()


def update(new_price, new_rating):
    global price, rating
    price = new_price
    rating = new_rating

widgets.interact(update,
                 new_price = widgets.IntSlider(min = 10, max = 50, value = price, description = 'Price'),
                 new_rating = widgets.FloatSlider(min = 3.5, max = 5, value = rating, description = 'Rating'));

interactive(children=(IntSlider(value=20, description='Price', max=50, min=10), FloatSlider(value=4.2, descrip…

In [162]:
plot_restaurants(seattle_restaurants, price, rating)

## Things to Consider

**Congratulations on building your (first?) machine learning algorithm!**

Here are a few things to think about, both now and when you interact with AI systems in the future:

* Did anything surprise you, either about the algorithm itself or its outputs?

* How could this system be applied or extended?

* What are the system's limitations? What could go wrong?

* What are some possible impacts of this algorithm on the world?